<a href="https://colab.research.google.com/github/OlenaBogoliubova/DATA-LOVE-ACADEMY/blob/main/HW_2_2_%D0%9D%D0%B5%D0%B7%D0%B1%D0%B0%D0%BB%D0%B0%D0%BD%D1%81%D0%BE%D0%B2%D0%B0%D0%BD%D0%B0_%D0%B1%D0%B0%D0%B3%D0%B0%D1%82%D0%BE%D0%BA%D0%BB%D0%B0%D1%81%D0%BE%D0%B2%D0%B0_%D0%BA%D0%BB%D0%B0%D1%81%D0%B8%D1%84%D1%96%D0%BA%D0%B0%D1%86%D1%96%D1%8F_Bogoliubova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTENC as SMOTENC_Tomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from sklearn.preprocessing import OrdinalEncoder
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
drive.mount('/content/drive')
customer_df = pd.read_csv("drive/MyDrive/DATA LOVE/customer_segmentation_train.csv")

print(customer_df.shape)
print(customer_df.head())
print(customer_df.info())
print(customer_df.isnull().sum())
print(customer_df['Segmentation'].value_counts())

Mounted at /content/drive
(8068, 11)
       ID  Gender Ever_Married  Age Graduated     Profession  Work_Experience  \
0  462809    Male           No   22        No     Healthcare              1.0   
1  462643  Female          Yes   38       Yes       Engineer              NaN   
2  466315  Female          Yes   67       Yes       Engineer              1.0   
3  461735    Male          Yes   67       Yes         Lawyer              0.0   
4  462669  Female          Yes   40       Yes  Entertainment              NaN   

  Spending_Score  Family_Size  Var_1 Segmentation  
0            Low          4.0  Cat_4            D  
1        Average          3.0  Cat_4            A  
2            Low          1.0  Cat_6            B  
3           High          2.0  Cat_6            B  
4           High          6.0  Cat_6            A  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------

In [ ]:
# Delete column ID
customer_df = customer_df.drop(columns=['ID'])

In [ ]:
# Preprocessing missing values
# Numerical - fill in median
customer_df['Work_Experience'] = customer_df['Work_Experience'].fillna(customer_df['Work_Experience'].median())
customer_df['Family_Size']     = customer_df['Family_Size'].fillna(customer_df['Family_Size'].median())

In [ ]:
# Categorical fill in mode
customer_df['Ever_Married'] = customer_df['Ever_Married'].fillna(customer_df['Ever_Married'].mode()[0])
customer_df['Graduated']    = customer_df['Graduated'].fillna(customer_df['Graduated'].mode()[0])
customer_df['Profession']   = customer_df['Profession'].fillna(customer_df['Profession'].mode()[0])
customer_df['Var_1']        = customer_df['Var_1'].fillna(customer_df['Var_1'].mode()[0])

In [ ]:
print("After preprocessing:")
print(customer_df.isnull().sum())

After preprocessing:
Gender             0
Ever_Married       0
Age                0
Graduated          0
Profession         0
Work_Experience    0
Spending_Score     0
Family_Size        0
Var_1              0
Segmentation       0
dtype: int64


In [ ]:
# Segmentation train/test
target_col = 'Segmentation'
input_cols = [col for col in customer_df.columns if col != target_col]

train_df, test_df = train_test_split(customer_df, test_size=0.2, random_state=42, stratify=customer_df[target_col])

train_inputs, train_targets = train_df[input_cols], train_df[target_col]
test_inputs,  test_targets  = test_df[input_cols],  test_df[target_col]

In [ ]:
# Columns
numeric_cols     = train_inputs.select_dtypes(include=['number']).columns.tolist()
categorical_cols = train_inputs.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric columns: {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")



Numeric columns: ['Age', 'Work_Experience', 'Family_Size']
Categorical columns: ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Spending_Score', 'Var_1']


In [ ]:
# Pipeline
numeric_transformer = Pipeline(steps=[
    ('scaler', MinMaxScaler())
])
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer,     numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

print("\nSize of dataset:")
print(f"Train: {train_inputs.shape}")
print(f"Test:  {test_inputs.shape}")
print(f"\nClass distribution in train:")
print(train_targets.value_counts())



Size of dataset:
Train: (6454, 9)
Test:  (1614, 9)

Class distribution in train:
Segmentation
D    1814
A    1578
C    1576
B    1486
Name: count, dtype: int64


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [ ]:
X_train_numeric = train_inputs[numeric_cols]
X_test_numeric  = test_inputs[numeric_cols]

scaler = MinMaxScaler()
X_train_numeric_scaled = scaler.fit_transform(X_train_numeric)
X_test_numeric_scaled  = scaler.transform(X_test_numeric)

# Encoding Target
le = LabelEncoder()
y_train = le.fit_transform(train_targets)
y_test  = le.transform(test_targets)

print("Classes after encoding:", le.classes_)
print(f"\nClasses before SMOTE:")
print(pd.Series(y_train).value_counts())

Classes after encoding: ['A' 'B' 'C' 'D']

Classes before SMOTE:
3    1814
0    1578
2    1576
1    1486
Name: count, dtype: int64


In [ ]:
# SMOTE on numerical values
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_numeric_scaled, y_train)

print(f"\nClasses after SMOTE:")
print(pd.Series(y_train_smote).value_counts())
print(f"\nSize before SMOTE:   {X_train_numeric_scaled.shape}")
print(f"Size after SMOTE: {X_train_smote.shape}")


Classes after SMOTE:
0    1814
1    1814
2    1814
3    1814
Name: count, dtype: int64

Size before SMOTE:   (6454, 3)
Size after SMOTE: (7256, 3)


In [ ]:
# Transformer for SMOTENC: scales numeric, ordinal encodes categorical
numeric_only_transformer = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numeric_cols)
    ], remainder='passthrough'
)

X_train_for_smotenc = numeric_only_transformer.fit_transform(train_inputs)
X_test_for_smotenc  = numeric_only_transformer.transform(test_inputs)

# Indices of categoricals columns
cat_feature_indices = list(range(len(numeric_cols), X_train_for_smotenc.shape[1]))
print(f"Numerical columns ({len(numeric_cols)}): Indices 0-{len(numeric_cols)-1}")
print(f"Categorical columns ({len(categorical_cols)}): Indices {cat_feature_indices}")

# SMOTENC
smotenc = SMOTENC(categorical_features=cat_feature_indices, random_state=42)
X_train_smotenc, y_train_smotenc = smotenc.fit_resample(X_train_for_smotenc, y_train)

print(f"\nClasses after SMOTENC:")
print(pd.Series(y_train_smotenc).value_counts())
print(f"\nSize before SMOTENC:   {X_train_for_smotenc.shape}")
print(f"Size after: {X_train_smotenc.shape}")

Numerical columns (3): Indices 0-2
Categorical columns (6): Indices [3, 4, 5, 6, 7, 8]

Classes after SMOTENC:
0    1814
1    1814
2    1814
3    1814
Name: count, dtype: int64

Size before SMOTENC:   (6454, 9)
Size after: (7256, 9)


In [ ]:
# Transformer for SMOTENC: scales numeric, ordinal encodes categorical, corrected handle_unknown and added unknown_value
smotenc_preprocessor_tomek = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numeric_cols),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ]
)

X_train_for_smotenc_processed, y_train_for_smotenc_processed = smotenc_preprocessor_tomek.fit_transform(train_inputs), y_train

# Recalculate categorical feature indices for the reprocessed data
# Numeric features come first (length of numeric_cols), then encoded categorical features
cat_feature_indices_processed = list(range(len(numeric_cols), X_train_for_smotenc_processed.shape[1]))

# SMOTE-Tomek
smote_tomek = SMOTETomek(
    smote=SMOTENC(categorical_features=cat_feature_indices_processed, random_state=42),
    tomek=TomekLinks(sampling_strategy='all'),
    random_state=42
)

X_train_smote_tomek, y_train_smote_tomek = smote_tomek.fit_resample(
    X_train_for_smotenc_processed, y_train_for_smotenc_processed
)

print(f"Classes after SMOTE-Tomek:")
print(pd.Series(y_train_smote_tomek).value_counts())
print(f"Size after SMOTE-Tomek: {X_train_smote_tomek.shape}")

Classes after SMOTE-Tomek:
2    1476
3    1451
1    1398
0    1377
Name: count, dtype: int64
Size after SMOTE-Tomek: (5702, 9)


**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [ ]:
from sklearn.metrics import classification_report

# Use the preprocessor that correctly handles both numeric and categorical features
X_test_processed_for_models = smotenc_preprocessor_tomek.transform(test_inputs)

# The original training data, fully processed
X_train_original_processed = X_train_for_smotenc_processed

# Re-apply SMOTENC to the fully processed original training data
smotenc_corrected = SMOTENC(categorical_features=cat_feature_indices_processed, random_state=42)
X_train_smotenc_corrected, y_train_smotenc_corrected = smotenc_corrected.fit_resample(X_train_original_processed, y_train)

datasets = {
    'Original':     (X_train_original_processed, y_train),
    'SMOTENC':      (X_train_smotenc_corrected, y_train_smotenc_corrected),
    'SMOTE-Tomek':  (X_train_smote_tomek, y_train_smote_tomek)
}

# Model training
for name, (X_tr, y_tr) in datasets.items():
    model = OneVsRestClassifier(
        LogisticRegression(random_state=42, max_iter=1000)
    )
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_test_processed_for_models)

    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))


Model: Original
              precision    recall  f1-score   support

           A       0.39      0.39      0.39       394
           B       0.41      0.09      0.14       372
           C       0.47      0.64      0.55       394
           D       0.59      0.79      0.67       454

    accuracy                           0.49      1614
   macro avg       0.47      0.48      0.44      1614
weighted avg       0.47      0.49      0.45      1614


Model: SMOTENC
              precision    recall  f1-score   support

           A       0.41      0.39      0.40       394
           B       0.36      0.15      0.21       372
           C       0.48      0.63      0.54       394
           D       0.62      0.78      0.69       454

    accuracy                           0.50      1614
   macro avg       0.47      0.48      0.46      1614
weighted avg       0.47      0.50      0.47      1614


Model: SMOTE-Tomek
              precision    recall  f1-score   support

           A       0.4

Для порівняння я б обрала метрику macro avg F1-score, бо задача мультикласова (4 класи), класи відносно збалансовані та macro avg враховує кожен клас однаково, незалежно від його розміру.

Найкращою моделлю став SMOTENC з macro F1=0.46, але різниця мінімальна.


Головною причиною відсутності суттєвої різниуі вважаю те, що логістична регресія є лінійною моделлю, і вона просто не здатна знайти складні нелінійні межі між класами A, B, C, D в цих даних. Ресемплінг допомагає при незбалансованих класах, але тут класи майже збалансовані (23-28%), тому його ефект мінімальний.

Що ще можна додати: клас B має дуже низький recall (0.09-0.15), модель погано його розпізнає; клас D найкращий (F1=0.67-0.69), тому що, мабуть, найбільш лінійно відділимий. Загальна accuracy ~50% для 4 класів — трохи краще за випадкове вгадування (25%), але далеко від хорошого результату.

Я б спробувала для цієї задачі інші, нелінійні моделі, наприклад дерева рішень, які краще впораються з такими даними.